# 이진 분류 실험 ⑤ — GFP 모티프 피처 (graph feature preprocessor)
IBM Snap ML 의 `GraphFeaturePreprocessor` 로 거래마다 **시간 창 안의 패턴 모양 수**를 센다: fan-in/out, degree, scatter-gather, 시간 순서 사이클, 길이 제한 사이클, 금액 vertex stats. BlazingAML 이 XGBoost + GFP 로 HI-Large F1 58 을 낸 그 피처 (`sampling/reference.md` 6절).

**질문**: 07 의 서비스 피처 셋(`tod`, 트리 AP 0.634) 위에 모티프 수를 더하면 — 특히 트리가 못 보는 **CYCLE · SCATTER-GATHER** 의 빡빡한 컷 recall 과 전체 AP 가 얼마나 오르나.

**첫 시도에서 배운 것** (1절 탐색 결과):
- 출력 열은 75 (bin 열 47 + vertex stats 7 피처 × 4 = 송신·수신 계좌 × in·out). 열 → 패턴 매핑은 **패턴을 하나씩 켜서 열 수를 세는 방식**으로 확정한다
- GFP 는 **배치 안의 간선을 전부 그래프에 넣은 뒤** 피처를 계산하므로 같은 배치의 나중 간선(미래) 도 센다. 500만 건 배치면 며칠치 미래. → **같은 분(timestamp) 단위 배치**로 흘린다 = 우리 규칙(과거만, 같은 분 예외) 과 동일. 호출 약 24만 번이라 속도를 2절에서 다시 잰다
- 메모리: 200만 건 배치에 RSS +4.9GB, DuckDB 8GB 와 합쳐 한도 초과로 죽음 → 분 단위 배치(수백 건) + pyarrow 스트리밍 읽기로

**진행**: 1 설치·열 매핑 → 2 속도 측정(분 배치) → 3 전체 변환(분 배치, 200만 건 단위로 parquet, 재개 가능) → 4 합치기 + compact 7열 → 5 트리 `tod_compact` / `tod_full`

## 0. 작업 준비

In [1]:
import sys, json, time, gc, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import psutil

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [2]:
# ---- 실험 설정 ------------------------------------------------------------
RUN_NAME = 'gfp'
cfg = B.Config(dataset='HI-Large', smoke=False)
B.seed_all(cfg.seed)
OUT_DIR = cfg.out_dir(RUN_NAME)
GFP_DIR = cfg.feat_dir / 'gfp_chunks'; GFP_DIR.mkdir(exist_ok=True)
GFP_FULL, GFP_COMPACT = cfg.feat_dir / 'gfp_full.parquet', cfg.feat_dir / 'gfp_compact.parquet'
TX_BASE = cfg.feat_dir / 'tx_base.parquet'          # edge_id 순(= 시간순). timestamp 는 최초 거래 기준 초 (float32)
READ_CHUNK = 2_000_000            # parquet 에서 한 번에 읽는 행 수 (= part 하나)
WARMUP_ROWS = 1_500_000           # 재개 시 그래프 상태 복원용으로 먼저 흘리는 앞 구간 (> 24h 창)
DAY = 86400

# GFP 설정 — 시간 창은 GFP 논문(HI: fan/degree/cycle 1일, scatter-gather 6시간). bin 은 모티프 크기 구간
GFP_PARAMS = {
    'num_threads': 8, 'time_window': DAY,
    'vertex_stats': True, 'vertex_stats_cols': [4],                 # 입력 4번 열 = 금액
    'vertex_stats_feats': [0, 1, 2, 3, 4, 5, 6],                     # fan, degree, ratio, avg, sum, min, max
    'fan': True,    'fan_tw': DAY,    'fan_bins':    [2, 3, 4, 5, 6, 8, 10, 15],
    'degree': True, 'degree_tw': DAY, 'degree_bins': [2, 3, 4, 5, 6, 8, 10, 15],
    'scatter-gather': True, 'scatter-gather_tw': 6 * 3600, 'scatter-gather_bins': [2, 3, 4, 6, 8],
    'temp-cycle': True, 'temp-cycle_tw': DAY, 'temp-cycle_bins': [2, 3, 4, 5, 6],
    'lc-cycle': True,   'lc-cycle_tw': DAY,   'lc-cycle_len': 6, 'lc-cycle_bins': [2, 3, 4, 5, 6],
}
PATTERN_KEYS = ['fan', 'degree', 'scatter-gather', 'temp-cycle', 'lc-cycle', 'vertex_stats']   # snapml 출력 열 순서 (이 순서로 가정, 1절에서 검증)
assert TX_BASE.exists(), 'tx_base.parquet 이 없다 — 03/06 을 먼저'

## 1. 설치 + 열 → 패턴 매핑
패턴을 하나씩만 켜고 출력 열 수를 세면, 전체 출력에서 각 패턴이 차지하는 열 범위가 나온다 (snapml 은 켜진 패턴 순서대로 열을 붙임). in/out 순서는 fan-out 을 심은 합성 그래프로 확인

In [3]:
# 설치 (한 번). 서버 환경에 따라 --break-system-packages 필요
# !{sys.executable} -m pip install --break-system-packages --no-cache-dir snapml
import snapml
from snapml import GraphFeaturePreprocessor
# print('snapml', snapml.__version__)

In [4]:
# 합성 간선 (열: edge_id, src, dst, ts초, amount). 0 → 1..6 fan-out 6개 / 7..12 → 13 fan-in 6개 / 20→21→22→20 3-사이클 / 30→31→32→33→30 4-사이클 / 나머지 무작위
rng = np.random.default_rng(0); t0 = 1_000_000
edges  = [[0, k, 60 * k] for k in range(1, 7)] + [[6 + k, 13, 600 + 60 * k] for k in range(1, 7)]
edges += [[20, 21, 1500], [21, 22, 1560], [22, 20, 1620], [30, 31, 1700], [31, 32, 1760], [32, 33, 1820], [33, 30, 1880]]
edges += [[int(rng.integers(40, 80)), int(rng.integers(40, 80)), int(2000 + 30 * i)] for i in range(60)]
probe = np.array([[i, s, d, t0 + t, float(rng.integers(10, 1000))] for i, (s, d, t) in enumerate(edges)], dtype=np.float64)
FANOUT_ROWS, FANIN_ROWS, CYC3_ROWS, CYC4_ROWS = list(range(0, 6)), list(range(6, 12)), [12, 13, 14], [15, 16, 17, 18]

def n_extra_cols(keys_on):
    p = dict(GFP_PARAMS); 
    for k in PATTERN_KEYS: p[k] = (k in keys_on)
    g = GraphFeaturePreprocessor(); g.set_params(p); return g.fit_transform(probe).shape[1] - probe.shape[1]
counts = {k: n_extra_cols([k]) for k in PATTERN_KEYS}
total = n_extra_cols(PATTERN_KEYS)
print('패턴별 열 수:', counts, '| 전부 켬:', total, '| 합:', sum(counts.values()))
assert total == sum(counts.values()), '패턴 열 수가 더해지지 않음 — 순서/중복 가정 재검토'
# 열 범위 (켜진 순서대로 이어 붙는다고 가정)
RANGES, pos = {}, 0
for k in PATTERN_KEYS:
    RANGES[k] = (pos, pos + counts[k]); pos += counts[k]
RANGES

패턴별 열 수: {'fan': 16, 'degree': 16, 'scatter-gather': 5, 'temp-cycle': 5, 'lc-cycle': 5, 'vertex_stats': 28} | 전부 켬: 75 | 합: 75


{'fan': (0, 16),
 'degree': (16, 32),
 'scatter-gather': (32, 37),
 'temp-cycle': (37, 42),
 'lc-cycle': (42, 47),
 'vertex_stats': (47, 75)}

In [5]:
# in/out 순서 확인 — 전부 켠 출력에서 fan-out 간선(0→1..6) 과 fan-in 간선(7..12→13) 이 fan 범위의 어느 반쪽을 켜는지
gp_probe = GraphFeaturePreprocessor(); gp_probe.set_params(GFP_PARAMS)
out = gp_probe.fit_transform(probe)[:, probe.shape[1]:]
a, b = RANGES['fan']; half = (b - a) // 2
fan_first, fan_second = out[:, a:a + half], out[:, a + half:b]
print('fan 앞 반쪽 — fan-out 간선 합', fan_first[FANOUT_ROWS].sum(), ' fan-in 간선 합', fan_first[FANIN_ROWS].sum())
print('fan 뒤 반쪽 — fan-out 간선 합', fan_second[FANOUT_ROWS].sum(), ' fan-in 간선 합', fan_second[FANIN_ROWS].sum())
FAN_ORDER = ('out', 'in') if fan_first[FANOUT_ROWS].sum() > fan_first[FANIN_ROWS].sum() else ('in', 'out')
print('→ fan/degree 열 순서:', FAN_ORDER)
c, d = RANGES['temp-cycle']; e, f = RANGES['lc-cycle']
print('temp-cycle 열 — 3-사이클 간선:', out[CYC3_ROWS, c:d].sum(axis=0), ' 4-사이클:', out[CYC4_ROWS, c:d].sum(axis=0), ' 배경:', out[20:, c:d].sum())
print('lc-cycle 열   — 3-사이클 간선:', out[CYC3_ROWS, e:f].sum(axis=0), ' 4-사이클:', out[CYC4_ROWS, e:f].sum(axis=0), ' 배경:', out[20:, e:f].sum())

fan 앞 반쪽 — fan-out 간선 합 0.0  fan-in 간선 합 40.0
fan 뒤 반쪽 — fan-out 간선 합 40.0  fan-in 간선 합 0.0
→ fan/degree 열 순서: ('in', 'out')
temp-cycle 열 — 3-사이클 간선: [0. 5. 0. 0. 0.]  4-사이클: [0. 0. 7. 0. 0.]  배경: 15.0
lc-cycle 열   — 3-사이클 간선: [0. 5. 0. 0. 0.]  4-사이클: [0. 0. 7. 0. 0.]  배경: 24.0


In [6]:
# 열 이름 확정 — 패턴_방향_bin번호 / vertex stats 는 위치 이름 (피처 7 × 4 조합, 조합 순서는 미확인이라 vs_<feat>_<k>)
VS_FEATS = ['fan', 'deg', 'ratio', 'avg', 'sum', 'min', 'max']
GFP_NAMES = []
for k in PATTERN_KEYS:
    a, b = RANGES[k]; n = b - a
    if k in ('fan', 'degree'):
        half = n // 2; tag = 'fan' if k == 'fan' else 'deg'
        GFP_NAMES += [f'gfp_{tag}_{FAN_ORDER[0]}_b{i}' for i in range(half)] + [f'gfp_{tag}_{FAN_ORDER[1]}_b{i}' for i in range(half)]
    elif k == 'vertex_stats':
        n_feat = len(GFP_PARAMS['vertex_stats_feats']); per = n // n_feat
        GFP_NAMES += [f'gfp_vs_{VS_FEATS[GFP_PARAMS["vertex_stats_feats"][i // per]]}_{i % per}' for i in range(n)]
    else:
        tag = {'scatter-gather': 'sg', 'temp-cycle': 'tcyc', 'lc-cycle': 'lccyc'}[k]
        GFP_NAMES += [f'gfp_{tag}_b{i}' for i in range(n)]
assert len(GFP_NAMES) == total == len(set(GFP_NAMES))
GROUPS = {f'gfp_{g}_total': [c for c in GFP_NAMES if c.startswith(f'gfp_{g}_b')] for g in ['fan_in', 'fan_out', 'deg_in', 'deg_out', 'sg', 'tcyc', 'lccyc']}
COMPACT_FEATURES = list(GROUPS)
print(len(GFP_NAMES), '열 |', {k: len(v) for k, v in GROUPS.items()})
pd.DataFrame(out, columns=GFP_NAMES).loc[CYC3_ROWS + FANOUT_ROWS[:2], [c for c in GFP_NAMES if c.startswith(('gfp_tcyc', 'gfp_lccyc', 'gfp_fan_out'))]]

75 열 | {'gfp_fan_in_total': 8, 'gfp_fan_out_total': 8, 'gfp_deg_in_total': 8, 'gfp_deg_out_total': 8, 'gfp_sg_total': 5, 'gfp_tcyc_total': 5, 'gfp_lccyc_total': 5}


,gfp_fan_out_b0,gfp_fan_out_b1,gfp_fan_out_b2,gfp_fan_out_b3,gfp_fan_out_b4,gfp_fan_out_b5,gfp_fan_out_b6,gfp_fan_out_b7,gfp_tcyc_b0,gfp_tcyc_b1,gfp_tcyc_b2,gfp_tcyc_b3,gfp_tcyc_b4,gfp_lccyc_b0,gfp_lccyc_b1,gfp_lccyc_b2,gfp_lccyc_b3,gfp_lccyc_b4
12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
0,2.0,2.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,2.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. 속도 측정 — 앞 200만 건을 **같은 분 단위 배치**로
분마다 transform 한 번 (평균 750건). 호출 고정 비용이 크면 느려지므로 전체 소요 시간을 여기서 예측. 읽기는 pyarrow 스트리밍 (DuckDB 없이)

In [7]:
PF = pq.ParquetFile(TX_BASE); COLS = ['edge_id', 'src', 'dst', 'timestamp', 'amount_received']
def iter_chunks(start_row=0, n_rows=None, chunk=READ_CHUNK):
    """tx_base 를 edge_id 순으로 chunk 행씩 float64 [edge_id, src, dst, ts초, amount] 로. start_row 부터, n_rows 만큼"""
    seen, taken = 0, 0
    for batch in PF.iter_batches(batch_size=chunk, columns=COLS):
        n = batch.num_rows
        if seen + n <= start_row:
            seen += n; continue
        arr = np.column_stack([batch.column(c).to_numpy().astype(np.float64) for c in COLS])
        if seen < start_row:
            arr = arr[start_row - seen:]
        seen += n
        if n_rows is not None and taken + len(arr) > n_rows:
            arr = arr[:n_rows - taken]
        taken += len(arr); yield arr
        if n_rows is not None and taken >= n_rows:
            return
N_TOTAL = PF.metadata.num_rows

def transform_by_minute(gp, X, fitted):
    """X 를 같은 timestamp(분) 묶음으로 잘라 순서대로 transform. 반환: 피처 배열(float64, 입력 열 제외), fitted 상태"""
    ts = X[:, 3]; assert (np.diff(ts) >= 0).all(), '시간순 아님'
    bounds = np.flatnonzero(np.diff(ts) != 0) + 1; starts = np.r_[0, bounds]; ends = np.r_[bounds, len(X)]
    out = np.empty((len(X), len(GFP_NAMES)), np.float64)
    for s, e in zip(starts, ends):
        res = gp.transform(X[s:e]) if fitted else gp.fit_transform(X[s:e]); fitted = True
        out[s:e] = res[:, X.shape[1]:]
    return out, fitted, len(starts)

sample = next(iter_chunks(0, 2_000_000))
gp_t = GraphFeaturePreprocessor(); gp_t.set_params(GFP_PARAMS)
rss0 = psutil.Process().memory_info().rss / 1e9; t0 = time.time()
feat, _, n_calls = transform_by_minute(gp_t, sample, False)
dt = time.time() - t0; rss1 = psutil.Process().memory_info().rss / 1e9
print(f'200만 건 / {n_calls:,} 분 배치: {dt:.0f}s → {len(sample)/dt:,.0f} 건/s, 호출당 {dt/n_calls*1000:.1f}ms | 전체 {N_TOTAL:,} 건 예상 {N_TOTAL/(len(sample)/dt)/3600:.1f} 시간 | RSS {rss0:.1f} → {rss1:.1f} GB')
fdf = pd.DataFrame(feat, columns=GFP_NAMES)
print('0 이 아닌 비율 (패턴별 합계):'); print({k: round(float((fdf[v].sum(axis=1) > 0).mean()), 4) for k, v in GROUPS.items()})

200만 건 / 338 분 배치: 135s → 14,859 건/s, 호출당 398.2ms | 전체 179,702,079 건 예상 3.4 시간 | RSS 0.9 → 3.6 GB
0 이 아닌 비율 (패턴별 합계):
{'gfp_fan_in_total': 0.0588, 'gfp_fan_out_total': 0.1196, 'gfp_deg_in_total': 0.2214, 'gfp_deg_out_total': 0.2461, 'gfp_sg_total': 0.0001, 'gfp_tcyc_total': 0.0, 'gfp_lccyc_total': 0.0}


In [8]:
del sample, feat, fdf, gp_t; gc.collect(); B.trim('(측정 후)')

RSS 0.8 GB (측정 후)


## 3. 전체 변환 — 분 단위 배치, 200만 건마다 parquet (재개 가능)
- `gfp_chunks/part_0000.parquet` … 각 part = `edge_id` + 피처 75열(float16). 있는 part 는 건너뜀
- 재개 시 GFP 내부 그래프가 비어 있으므로 첫 미완 part 앞 150만 건(> 24h) 을 먼저 흘려 상태를 채운다 (출력 버림)
- 2절 예측 시간 참고. 중간에 멈추면 이 셀만 다시

In [ ]:
n_parts = (N_TOTAL + READ_CHUNK - 1) // READ_CHUNK
parts = [GFP_DIR / f'part_{i:04d}.parquet' for i in range(n_parts)]
first_missing = next((i for i, p in enumerate(parts) if not p.exists()), n_parts)
print(f'part {n_parts}개, 완료 {first_missing}개')
if first_missing < n_parts:
    gp = GraphFeaturePreprocessor(); gp.set_params(GFP_PARAMS); fitted = False
    t_all = time.time()
    if first_missing > 0:
        wstart = max(0, first_missing * READ_CHUNK - WARMUP_ROWS)
        for warm in iter_chunks(wstart, first_missing * READ_CHUNK - wstart):
            _, fitted, _ = transform_by_minute(gp, warm, fitted)
        del warm; print(f'  워밍업 완료 ({WARMUP_ROWS:,} 건)')
    for i, X in enumerate(iter_chunks(first_missing * READ_CHUNK), start=first_missing):
        t0 = time.time()
        feat, fitted, n_calls = transform_by_minute(gp, X, fitted)
        df = pd.DataFrame(feat.astype(np.float16), columns=GFP_NAMES); df.insert(0, 'edge_id', X[:, 0].astype(np.int64))
        df.to_parquet(parts[i], index=False, compression='zstd')
        del X, feat, df
        B.log(f'  part {i:04d}/{n_parts-1}  {time.time()-t0:.0f}s  ({n_calls:,} 호출)  누적 {(time.time()-t_all)/60:.0f}분  RSS {psutil.Process().memory_info().rss/1e9:.1f}GB')
    del gp; gc.collect()
print('완료')

part 90개, 완료 1개
  워밍업 완료 (1,500,000 건)


/tmp/ipykernel_801402/3335661697.py:16: RuntimeWarning: overflow encountered in cast
  df = pd.DataFrame(feat.astype(np.float16), columns=GFP_NAMES); df.insert(0, 'edge_id', X[:, 0].astype(np.int64))


[15:39:06]   part 0001/89  771s  (895 호출)  누적 14분  RSS 3.3GB


## 4. 합치기 — `gfp_full.parquet` (75열) + `gfp_compact.parquet` (패턴별 bin 합계 7열, ln(1+x))

In [ ]:
con = L.connect(threads=8, memory='4GB')
if not GFP_FULL.exists():
    t0 = time.time()
    con.execute(f"COPY (SELECT * FROM read_parquet({L.q(str(GFP_DIR / 'part_*.parquet'))}) ORDER BY edge_id) TO {L.q(GFP_FULL)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    print(f'gfp_full.parquet {time.time()-t0:.0f}s')
n_full = con.execute(f"SELECT count(*) FROM read_parquet({L.q(GFP_FULL)})").fetchone()[0]
assert n_full == N_TOTAL, f'행 수 불일치 {n_full:,} vs {N_TOTAL:,}'
if not GFP_COMPACT.exists():
    dq = lambda col: '"' + col + '"::DOUBLE'
    exprs = [f"ln(1 + ({' + '.join(dq(col) for col in cols)}))::FLOAT AS {name}" for name, cols in GROUPS.items()]
    t0 = time.time()
    con.execute(f"COPY (SELECT edge_id, {', '.join(exprs)} FROM read_parquet({L.q(GFP_FULL)}) ORDER BY edge_id) TO {L.q(GFP_COMPACT)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    print(f'gfp_compact.parquet {time.time()-t0:.0f}s')
con.execute(f"SELECT count(*), {', '.join(f'avg({c}) AS {c[4:]}' for c in COMPACT_FEATURES)} FROM read_parquet({L.q(GFP_COMPACT)})").df().T

In [ ]:
# 패턴 라벨별 평균 (테스트 구간) — CYCLE 거래에서 사이클 열이, SCATTER-GATHER 에서 sg 열이 큰지
con.execute(f"""
    SELECT coalesce(p.pattern_type, CASE WHEN t.is_laundering THEN 'UNMATCHED' ELSE 'NORMAL' END) AS grp, count(*) AS n,
           {', '.join(f'avg(g.{c})::FLOAT AS {c[4:]}' for c in COMPACT_FEATURES)}
    FROM read_parquet({L.q(cfg.tx_path)}) t JOIN read_parquet({L.q(GFP_COMPACT)}) g USING (edge_id)
    LEFT JOIN (SELECT DISTINCT edge_id, pattern_type FROM read_parquet({L.q(L.BASE / 'patterns.parquet')})) p USING (edge_id)
    WHERE t.day BETWEEN DATE '2022-10-17' AND DATE '2022-10-30'
    GROUP BY 1 ORDER BY n DESC""").df().set_index('grp').round(3)

## 5. 트리 — `tod` + GFP
07 과 같은 구조. 조건: `tod_compact` (30 + 7), `tod_full` (30 + 75). `tod` 기준선은 07 결과를 읽어 비교

In [ ]:
currency_codes, format_codes = B.code_tables(con, cfg)
PAIR_FEATURES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'reverse_history_count']
ACT_FEATURES  = ['src_out_1h', 'src_out_1d', 'src_uniq_dst_1d', 'src_out_amt_1d', 'dst_in_1h', 'dst_in_1d', 'dst_uniq_src_1d', 'dst_in_amt_1d']
HIST_FEATURES = ['src_hist_out', 'src_hist_uniq_out', 'src_active_days', 'src_rate_vs_prior', 'src_minute_burst', 'src_gap_minutes',
                 'dst_hist_in', 'dst_hist_uniq_in', 'dst_active_days', 'dst_rate_vs_prior', 'dst_minute_burst', 'dst_gap_minutes']
TOD_FEATURES  = ['hour_sin', 'hour_cos', 'weekday']
PATHS = {'pair': cfg.feat_dir / 'pair_history.parquet', 'act': cfg.feat_dir / 'account_activity.parquet',
         'hist': cfg.feat_dir / 'account_history.parquet', 'tod': cfg.feat_dir / 'time_of_day.parquet'}
TOD_SET = [c for c in B.BASIC_EDGE_FEATURES if c != 'timestamp'] + TOD_FEATURES + PAIR_FEATURES + ACT_FEATURES + HIST_FEATURES   # 07 의 tod 조건 (30)
CONDITIONS = {'tod_compact': TOD_SET + COMPACT_FEATURES, 'tod_full': TOD_SET + GFP_NAMES}
ALL_FEATURES = TOD_SET + COMPACT_FEATURES + GFP_NAMES
{k: len(v) for k, v in CONDITIONS.items()}, len(ALL_FEATURES)

In [ ]:
con.close(); con = L.connect(threads=8, memory='3GB')
con.register('currency_codes', currency_codes); con.register('format_codes', format_codes)
tx = B.load_tx_lowmem(con, cfg); tx.drop(columns=['timestamp'], inplace=True)
B.trim('(tx 적재 후)')
tx = B.attach_parquet(tx, PATHS['tod'],  TOD_FEATURES,  dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['pair'], PAIR_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['act'],  ACT_FEATURES,  dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['hist'], HIST_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, GFP_COMPACT, COMPACT_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, GFP_FULL, GFP_NAMES, dtype=np.float16)          # 이미 float16 로 저장돼 있음
B.trim(f'(피처 {len(ALL_FEATURES)}열 붙인 후)'); tx.shape

In [ ]:
in_train = B.window_mask(tx, cfg.train_window)
FEAT_PATH = cfg.feat_dir / f'feat16_{RUN_NAME}.npy'
feat_z, znorm_table = B.znorm_features(tx, ALL_FEATURES, in_train, out_path=FEAT_PATH)
B.trim('(z-norm 후)'); znorm_table.tail(12)

In [ ]:
val_df = B.val_sample(con, cfg.val_window, cfg.val_neg_rate)
test_df = B.test_full(con, cfg.test_window)
in_degree = B.recipient_indegree(con, cfg)
con.close()
all_edge_ids = tx['edge_id'].to_numpy()
val_idx = B.rows_of(all_edge_ids, val_df['edge_id'].to_numpy()); test_idx = B.rows_of(all_edge_ids, test_df['edge_id'].to_numpy())
assert (tx['y'].to_numpy()[val_idx] == val_df['y'].to_numpy()).all()
del all_edge_ids
train_rows = np.flatnonzero(in_train); y_train = tx['y'].to_numpy()[in_train]
y_val, w_val = val_df['y'].to_numpy(), val_df['w'].to_numpy()
del tx; gc.collect(); B.trim('(tx 삭제 후)')

In [ ]:
def run_condition(name, feats):
    out = OUT_DIR / name; out.mkdir(exist_ok=True)
    if (out / 'compare.csv').exists():
        print(f'[{name}] 이미 완료 — 건너뜀'); return
    B.log(f'[{name}] 피처 {len(feats)}')
    cols = [ALL_FEATURES.index(c) for c in feats]
    X_val = B.rows_matrix(feat_z, val_idx, cols)
    test_pred = lambda m: B.predict_rows(m, feat_z, test_idx, cols)
    results = {}
    lgb_model, xgb_model = B.train_trees_lowmem(feat_z, train_rows, cols, y_train, X_val, y_val, w_val, test_pred, val_df, test_df, cfg, results)
    imp = pd.DataFrame({'feature': feats, 'xgboost_gain': [xgb_model.get_booster().get_score(importance_type='gain').get(f'f{i}', 0.0) for i in range(len(feats))],
                        'lightgbm_gain': lgb_model.booster_.feature_importance('gain')})
    imp.sort_values('xgboost_gain', ascending=False).to_csv(out / 'importance.csv', index=False, encoding='utf-8-sig')
    pattern_tables, degree_tables, cut_tables = [], [], []
    for mname, model in [('xgboost', xgb_model), ('lightgbm', lgb_model)]:
        sc = pd.DataFrame(dict(edge_id=test_df.edge_id, y=test_df.y, score=test_pred(model)))
        sc.to_parquet(out / f'{mname}_test_scores.parquet', index=False)
        pattern_tables.append(B.pattern_recall(sc).assign(model=mname))
        degree_tables.append(B.degree_stratified(sc, test_df['dst'].to_numpy(), in_degree).assign(model=mname))
        cut_tables.append(B.cut_table(sc).assign(model=mname))
    pd.concat(pattern_tables).to_csv(out / 'analysis_pattern.csv', encoding='utf-8-sig')
    pd.concat(degree_tables).to_csv(out / 'analysis_degree.csv', encoding='utf-8-sig')
    pd.concat(cut_tables).to_csv(out / 'analysis_cuts.csv', encoding='utf-8-sig')
    B.save_results(out, cfg, results, features=dict(tree=feats))
    del X_val, lgb_model, xgb_model; gc.collect(); B.trim(f'[{name}] 끝')

for name, feats in CONDITIONS.items():
    run_condition(name, feats)

## 6. 비교 — `tod`(07) vs `tod_compact` vs `tod_full`

In [ ]:
BASE = Path.cwd() / 'results' / cfg.dataset / 'tree_ablations' / 'tod'       # 07 의 tod 조건
RUNS = {'tod (07)': BASE, **{n: OUT_DIR / n for n in CONDITIONS if (OUT_DIR / n / 'compare.csv').exists()}}
rows = []
for name, d in RUNS.items():
    c = pd.read_csv(d / 'compare.csv', index_col=[0, 1])
    for model in c.index.get_level_values(0).unique():
        t = c.loc[(model, 'test')]; v = c.loc[(model, 'val')]
        rows.append(dict(condition=name, model=model, val_AP=v['AP'], test_AP=t['AP'], F1=t['F1'], precision=t['precision'], recall=t['recall'],
                         recall_1pct=t['recall_at_1pct'], unmatched_1pct=t['unmatched_recall_at_1pct']))
summary = pd.DataFrame(rows).set_index(['model', 'condition']).sort_index()
summary.to_csv(OUT_DIR / 'summary.csv', encoding='utf-8-sig'); summary.round(4)

In [ ]:
# 패턴별 recall — r@0.1% 와 argmax (XGBoost). 질문: CYCLE · SCATTER-GATHER 가 오르나
pat = pd.concat([pd.read_csv(d / 'analysis_pattern.csv', index_col=0).query("model == 'xgboost'").assign(condition=n) for n, d in RUNS.items()])
order = [p for p in B.PATTERN_ORDER + ['ALL'] if p in pat.index]
display(pat.reset_index().pivot(index='pattern_type', columns='condition', values='r@0.1%').loc[order, list(RUNS)].round(4))
pat.reset_index().pivot(index='pattern_type', columns='condition', values='argmax').loc[order, list(RUNS)].round(4)

In [ ]:
# 수신 계좌 in-degree 구간별 AP (XGBoost)
deg = pd.concat([pd.read_csv(d / 'analysis_degree.csv', index_col=0).query("model == 'xgboost'").assign(condition=n) for n, d in RUNS.items()])
deg.reset_index().pivot(index='bin', columns='condition', values='AP')[list(RUNS)].round(4)

In [ ]:
# GFP 피처 중요도 — tod_full 에서 상위 15 (XGBoost gain). 어떤 모티프 열이 쓰였나
imp = pd.read_csv(OUT_DIR / 'tod_full' / 'importance.csv'); imp['is_gfp'] = imp.feature.str.startswith('gfp_')
display(imp.head(15)); print('GFP 열의 gain 비중:', round(imp.loc[imp.is_gfp, 'xgboost_gain'].sum() / imp.xgboost_gain.sum(), 3))

## 7. 읽는 법
- `tod_full − tod`: 모티프 수의 전체 몫. BlazingAML 이 GFP 로 얻은 이득이 우리 피처 위에서도 남아 있나
- CYCLE · SCATTER-GATHER 의 r@0.1% · argmax 변화: 트리가 못 보던 2홉·사이클 구조가 피처로 들어온 효과
- `tod_compact` 가 `tod_full` 에 가깝다면 7열로 충분 → GINe 에도 붙일 수 있음 (다음 런)